# YOLO26s 3차 실험 (Colab)

2차 v3 e20 위에서 클래스 불균형 대응 세 가지를 돌림. 코드는 팀 `exp3/exp3_ultralytics_smoke_test.ipynb` 기준이고, 설정은 2차 그대로.

| 조건 | train 목록 | 손실 가중치 |
|---|---|---|
| sampling | RFS 오버샘플링 | 없음 |
| weight | 원래 v3 | cls_pw 0.5 |
| both | RFS 오버샘플링 | cls_pw 0.5 |

비교 기준은 2차 `v3_yolo26s_e20_s0` (Kaggle 0.6339). 다시 학습 안 함.

## 1. 설정, 세션 시작

In [ ]:
# 바꿀 값. 바꾸면 이 셀부터 다시 실행
VARIANT = 'v3'
MODEL = 'yolo26s'
BASE_EPOCHS = 20                                            # 비교할 2차 run의 에폭
BASE_EPOCH_ARGS = dict(close_mosaic=10, warmup_epochs=3.0)  # 2차 때 기본값 그대로
# 2차 train_one이랑 같은 값
TRAIN_ARGS = dict(optimizer='AdamW', lr0=0.000167, momentum=0.9, warmup_bias_lr=0.0,
                  imgsz=640, batch=16, seed=0, patience=30, workers=8, device=0, cache=False)

SAMPLING = 'rfs'   # 오버샘플링
CLS_PW = 0.5       # 손실 가중치 세기 (팀 기본값)
CONDITIONS = ['sampling', 'weight', 'both']
SMOKE = False      # 먼저 True로 확인, 본 학습은 False

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/pill data')
SAVE_DIR = DRIVE_BASE / 'runs'      # 1·2차랑 같은 결과 폴더
REPO = '/content/pill-object-detection'
BRANCH = 'main'

if not SAVE_DIR.is_dir():  # 경로 틀려도 ultralytics는 새 폴더를 만들어버림
    raise FileNotFoundError(f'드라이브에 결과 폴더가 없음: {SAVE_DIR}')

# 팀 레포 받기 (이미 있으면 최신으로만)
!test -d "{REPO}" || git clone -q https://github.com/codeit-ai-15th/pill-object-detection.git "{REPO}"
%cd {REPO}
!git fetch -q origin {BRANCH} && git switch -q -C {BRANCH} FETCH_HEAD && git log -1 --format='%h %s'
sys.path.insert(0, f'{REPO}/src')

# 2차랑 같은 버전 + 제출용 kaggle
!pip install -q ultralytics==8.4.183 pycocotools==2.0.11 wandb==0.31.0 kaggle

import wandb
os.environ['WANDB_ENTITY'] = 'codeit-prj1-team2'   # 로그인보다 먼저
wandb.login(key=userdata.get('WANDB_API_KEY'))     # 키는 2차처럼 보안 비밀에서

import torch, ultralytics
print('ultralytics', ultralytics.__version__, '| GPU', torch.cuda.get_device_name(0))

## 2. 데이터, 조건별 학습 계획
v3 학습 폴더를 만들고, RFS 목록이랑 조건별 에폭을 정함. 에폭·close_mosaic·warmup은 총 학습량이 2차 v3 e20과 같도록 목록 길이에 맞춰 줄임.

In [ ]:
# 세션마다. 이미 만들었으면 건너뜀
DS = f'/content/exp2/{VARIANT}'
if not Path(DS).is_dir():
    for z in ['kaggle', 'aihub_annotations', 'aihub_images']:
        !unzip -q -o "{DRIVE_BASE}/data/{z}.zip" -d {REPO}
    !PYTHONPATH=src python scripts/build_dataset.py --variant {VARIANT} --out {DS}

!ls {DS}/images/train | wc -l   # v3면 8974

처음 실행 때 출력: 클래스 96개, train 8,974장 / val 1,000장, 객체 train 33,872개 / val 3,790개 (2차 v3와 같음). 본 학습 때 smoke를 false로 바꾸고 다시 실행하면서 폴더 만드는 부분을 건너뛰어 출력이 덮어써짐.

In [ ]:
import pandas as pd
from pill_detection.balance import load_train_objects, repeat_list, write_train_list

objects = load_train_objects(DS)
files = repeat_list(objects, seed=0)               # 희소 클래스가 든 사진을 더 반복
info = write_train_list(DS, SAMPLING, files)       # DS/sampling/rfs/에 train.txt, data.yaml

def plan(tag, data, n_rows, cls_pw):
    # 목록 길이가 바뀐 만큼 에폭 단위 설정을 같은 비율로 바꿔서 총 step을 2차 run과 맞춤
    scale = info['n_base'] / n_rows
    return {'name': f'{VARIANT}_{MODEL}_{tag}', 'data': data, 'n_rows': n_rows, 'cls_pw': cls_pw,
            'epochs': round(BASE_EPOCHS * scale),
            'close_mosaic': round(BASE_EPOCH_ARGS['close_mosaic'] * scale),
            'warmup_epochs': round(BASE_EPOCH_ARGS['warmup_epochs'] * scale, 2)}

PLANS = {
    'sampling': plan(SAMPLING, f'{DS}/sampling/{SAMPLING}/data.yaml', info['n_list'], 0.0),
    'weight': plan(f'cls{CLS_PW}', f'{DS}/data.yaml', info['n_base'], CLS_PW),
    'both': plan(f'{SAMPLING}_cls{CLS_PW}', f'{DS}/sampling/{SAMPLING}/data.yaml', info['n_list'], CLS_PW),
}
print('원래 train', info['n_base'], '장 | RFS 목록', info['n_list'], '줄')
display(pd.DataFrame(PLANS).T.loc[CONDITIONS, ['name', 'n_rows', 'epochs', 'close_mosaic', 'warmup_epochs', 'cls_pw']])

## 3. 학습과 채점
조건마다 학습하고 best.pt로 val 채점. 다시 실행하면 채점까지 끝난 조건은 건너뛰고, 끊긴 run은 last.pt에서 이어감.
학습 직후 train 장수, 클래스 가중치 유무, close_mosaic·warmup이 계획표와 같은지 확인 출력.

In [ ]:
import json, shutil
from pill_detection.balance import imbalance_summary

def find_run_id(run_dir):
    # 학습 때 W&B 실행 id 찾기 (여러 개면 최근 것)
    found = max(Path(run_dir, 'wandb').glob('*run-*/run-*.wandb'), key=lambda f: f.stat().st_mtime, default=None)
    if found is None:
        raise FileNotFoundError(f'W&B 기록이 없음: {run_dir}/wandb/')
    return found.stem[4:]

def save_scores(run_dir, scores, preds, condition):
    # val 점수 + 희소/빈번 절반 AP 차이(gap)를 찍고 드라이브랑 W&B에 남김
    summary = {k: scores[k] for k in ('val', 'kaggle', 'test_only')} | imbalance_summary(scores['per_class'], f'{DS}/classes.csv', VARIANT)
    print(Path(run_dir).name, '|', '  '.join(f'{k} {v:.4f}' for k, v in summary.items() if k != 'n_classes'))
    Path(run_dir, 'scores.json').write_text(json.dumps(scores, indent=2), encoding='utf-8')
    Path(run_dir, 'val_predictions.json').write_text(json.dumps(preds), encoding='utf-8')  # confusion matrix용 (2차랑 같게)
    Path(run_dir, 'condition.json').write_text(json.dumps(condition, indent=2), encoding='utf-8')
    shutil.copy(f'{DS}/dataset_info.json', run_dir)
    with wandb.init(entity='codeit-prj1-team2', project='pill-object-detection', id=find_run_id(run_dir),
                    resume='allow', dir=str(run_dir)) as run:
        run.summary.update({f'exp3/{k}': v for k, v in summary.items()})
        run.config.update({'exp3': condition})

In [ ]:
from datetime import datetime
from ultralytics import YOLO, settings
from pill_detection.evaluate import evaluate_val, predictions_from_ultralytics

settings.update({'wandb': True})

def tag_wandb(tags):
    # 첫 에폭 직전. 팀 W&B에 안 붙었으면 멈추고, 붙었으면 태그 달기
    if wandb.run is None or wandb.run.entity != 'codeit-prj1-team2':
        raise RuntimeError('W&B 기록이 팀 프로젝트에 시작 안 됨. 런타임 다시 시작하고 맨 위부터')
    wandb.run.tags = sorted({*wandb.run.tags, *tags})

def check_applied(trainer, p):
    # 조건이 학습에 그대로 들어갔는지
    n_train, expected = len(trainer.train_loader.dataset), max(1, round(p['n_rows'] * trainer.args.fraction))
    weighted = getattr(trainer.model, 'class_weights', None) is not None
    schedule = {k: getattr(trainer.args, k) for k in ('close_mosaic', 'warmup_epochs')}
    print(f"{trainer.args.name}: train {n_train:,}장 (예상 {expected:,}), 클래스 가중치 {'있음' if weighted else '없음'}, "
          f"close_mosaic {schedule['close_mosaic']} (예상 {p['close_mosaic']}), warmup_epochs {schedule['warmup_epochs']} (예상 {p['warmup_epochs']})")
    if n_train != expected or weighted != bool(p['cls_pw']) or any(schedule[k] != p[k] for k in schedule):
        raise RuntimeError('조건이 학습에 그대로 안 들어감. PLANS랑 학습 인자 확인')

for condition in CONDITIONS:
    p = PLANS[condition]
    epochs = 1 if SMOKE else p['epochs']
    run_dir = SAVE_DIR / (f"smoke_{p['name']}_{datetime.now():%m%d_%H%M}" if SMOKE else p['name'])
    if (run_dir / 'scores.json').exists():
        print(f'{run_dir.name}: 채점까지 끝나서 건너뜀')
        continue
    results_csv = run_dir / 'results.csv'
    if not (results_csv.exists() and len(results_csv.read_text().splitlines()) - 1 >= epochs):
        last = run_dir / 'weights' / 'last.pt'   # 있으면 끊긴 run
        model = YOLO(last if last.exists() else f'{MODEL}.pt')
        tags = ['exp3', VARIANT, condition] + (['smoke'] if SMOKE else [])
        model.add_callback('on_train_start', lambda trainer: tag_wandb(tags))
        if last.exists():
            model.train(resume=True)
        else:
            model.train(data=p['data'], cls_pw=p['cls_pw'], epochs=epochs, close_mosaic=p['close_mosaic'],
                        warmup_epochs=p['warmup_epochs'], fraction=0.05 if SMOKE else 1.0,
                        project='pill-object-detection', save_dir=str(run_dir), name=run_dir.name, **TRAIN_ARGS)
        check_applied(model.trainer, p)
    results = YOLO(run_dir / 'weights' / 'best.pt').predict(f'{DS}/images/val', conf=0.001, stream=True, verbose=False)
    preds = predictions_from_ultralytics(results)
    save_scores(run_dir, evaluate_val(preds, f'{DS}/eval/val_gt.json'), preds,
                {'condition': condition, **p, 'sampling': SAMPLING, 'base_epochs': BASE_EPOCHS, 'smoke': SMOKE})

#### 결과: 3차 본 학습 (val)

| 조건 | run | val | kaggle 47장 | test_only | rare_ap | frequent_ap | gap |
|---|---|---|---|---|---|---|---|
| sampling | v3_yolo26s_rfs | 0.9936 | 0.9973 | 0.9916 | 0.9938 | 0.9933 | -0.0005 |
| weight | v3_yolo26s_cls0.5 | 0.9936 | 0.9896 | 0.9939 | 0.9935 | 0.9937 | 0.0002 |
| both | v3_yolo26s_rfs_cls0.5 | 0.9921 | 0.9987 | 0.9890 | 0.9924 | 0.9919 | -0.0006 |

- 2차 기준 v3 e20: val 0.9918, kaggle 47장 0.9905
- 세 조건 모두 계획대로 들어감 (train 장수, 클래스 가중치 유무, close_mosaic, warmup 확인 통과)
- A100, 조건당 학습+채점 약 30분
- val은 세 조건 다 0.992~0.994로 2차 기준이랑 0.002 안. 2차처럼 포화돼 있어서 val로는 차이를 못 봄
- gap(train 객체 많은 절반 AP - 적은 절반 AP, +면 희소 클래스가 더 약함)은 -0.0006~0.0002로 거의 0. 2차에서도 클래스별 AP와 train 수 상관이 0.08이라, val에는 처음부터 고칠 불균형이 거의 안 보였음
- 효과는 Kaggle 제출로 봐야 함

In [ ]:
# 3차 세 조건 vs 2차 기준 (v3 e20) val 점수 비교
from pill_detection.balance import imbalance_summary

RUNS = {'2차 기준': 'v3_yolo26s_e20_s0',
        'sampling': 'v3_yolo26s_rfs',
        'weight': 'v3_yolo26s_cls0.5',
        'both': 'v3_yolo26s_rfs_cls0.5'}

rows = {}
for label, name in RUNS.items():
    s = json.loads((SAVE_DIR / name / 'scores.json').read_text())
    res = pd.read_csv(SAVE_DIR / name / 'results.csv')
    rows[label] = {'run': name, '에폭': len(res),
                   **{k: s[k] for k in ('val', 'kaggle', 'test_only')},
                   **imbalance_summary(s['per_class'], f'{DS}/classes.csv', VARIANT)}

cmp = pd.DataFrame(rows).T.drop(columns='n_classes')
display(cmp.round(4))

#### 결과: 2차 기준 vs 3차 (val)

| 조건 | 에폭 | val | kaggle 47장 | test_only | rare_ap | frequent_ap | gap |
|---|---|---|---|---|---|---|---|
| 2차 기준 (v3 e20) | 20 | 0.9918 | 0.9905 | 0.9909 | 0.9915 | 0.9920 | +0.0005 |
| sampling | 18 | 0.9936 | 0.9973 | 0.9916 | 0.9938 | 0.9933 | -0.0005 |
| weight | 20 | 0.9936 | 0.9896 | 0.9939 | 0.9935 | 0.9937 | +0.0002 |
| both | 18 | 0.9921 | 0.9987 | 0.9890 | 0.9924 | 0.9919 | -0.0006 |

- 2차 기준도 gap +0.0005로 희소 클래스가 약하다고 할 만한 게 거의 없었음
- sampling, both는 gap이 -로 바뀌어서 방향은 맞지만, 크기가 0.001 단위라 val로는 효과라고 보기 어려움
- val 차이는 전부 0.002 안. 순위도 지표마다 엇갈림 (val은 weight, kaggle 47장은 both가 제일 높음)
- 2차에서 val 순위랑 Kaggle 순위가 안 맞았으니, 판단은 Kaggle 제출로

## 4. Kaggle 제출
세 조건 다 제출. val로는 못 골라서 Kaggle로 비교함. 제출 설정은 2차랑 같게 conf 0.01.

In [ ]:
import subprocess
from google.colab import userdata

COMP = 'ai15-level1-project'
SUB_DIR = SAVE_DIR / 'submissions'   # 1·2차 제출 csv랑 같은 곳

def kaggle_login():
    # 보안 비밀의 kaggle.json을 파일로 (세션마다)
    cred = json.loads(userdata.get('KAGGLE_JSON'))
    d = Path.home() / '.kaggle'
    d.mkdir(exist_ok=True)
    (d / 'kaggle.json').write_text(json.dumps(cred))
    os.chmod(d / 'kaggle.json', 0o600)

def run_kaggle(args):
    # 결과를 셀에 찍기 (그냥 실행하면 안 보임)
    r = subprocess.run(['kaggle', 'competitions', *args], capture_output=True, text=True)
    print(r.stdout or r.stderr)
    r.check_returncode()

kaggle_login()
run_kaggle(['submissions', '-c', COMP])   # 지난 제출 목록이 보이면 연결된 것

In [ ]:
from pill_detection.evaluate import write_submission, competition_from_classes

# test 사진 풀기
!unzip -q -o "{DRIVE_BASE}/data/test_images.zip" -d "{REPO}"
!ls {REPO}/data/raw/test_images | wc -l   # 842면 OK

COMPETITION = competition_from_classes(f'{REPO}/data/datasets/classes.csv')
print('대회 클래스:', len(COMPETITION))   # 74개여야 함
TEST_DIR = f'{REPO}/data/raw/test_images'

SUBMIT_RUNS = ['v3_yolo26s_rfs', 'v3_yolo26s_cls0.5', 'v3_yolo26s_rfs_cls0.5']
for name in SUBMIT_RUNS:
    out = SUB_DIR / f'exp3_{name}.csv'
    assert not out.exists(), f'{out.name} 이미 있음. 덮어쓰지 않고 멈춤'
    results = YOLO(SAVE_DIR / name / 'weights' / 'best.pt').predict(TEST_DIR, conf=0.01, device=0, stream=True, verbose=False)
    n = write_submission(predictions_from_ultralytics(results), out, COMPETITION)
    print(f'{name}: {n}줄 → {out.name}')

In [ ]:
# 세 개 제출하고 점수 확인
for name in SUBMIT_RUNS:
    run_kaggle(['submit', '-c', COMP, '-f', str(SUB_DIR / f'exp3_{name}.csv'), '-m', f'exp3 {name}'])

import time
time.sleep(60)   # 채점 기다리기
run_kaggle(['submissions', '-c', COMP])

#### 결과: Kaggle 제출 (11/4)

| 조건 | run | Kaggle public | 2차 기준 대비 |
|---|---|---|---|
| 2차 기준 | v3_yolo26s_e20_s0 | 0.6339 | |
| both | v3_yolo26s_rfs_cls0.5 | 0.6198 | -0.014 |
| weight | v3_yolo26s_cls0.5 | 0.6126 | -0.021 |
| sampling | v3_yolo26s_rfs | 0.5985 | -0.035 |

- 세 조건 다 2차 기준보다 낮음. 불균형 대응으로 더 좋아지지 않음
- sampling은 흔들림(0.02)을 넘게 떨어짐. weight는 경계, both는 흔들림 안
- val은 세 조건 다 2차 기준보다 살짝 높았는데 Kaggle은 다 낮음. 2차처럼 val로는 순서를 못 맞춤
- val에서 처음부터 희소 클래스가 약하지 않았으니(gap +0.0005), 고칠 게 없는 데를 건드린 셈
- 최종 후보는 2차 v3 e20 그대로
- seed 0 한 번씩이라 0.02 안쪽 차이는 확실하지 않음. 기준 v3 e20도 e10보다 +0.03 뛴 run이라 높게 나왔을 수 있음

## 5. 추가 실험: seed 반복, 모자이크 분리
2차 v3 e20(0.6339)이 seed 하나짜리고, e10보다 +0.03 뛴 이유가 에폭인지 모자이크인지 못 나눠서 추가로 돌림. 설정은 2차 v3 e20 그대로, 바꾼 것만 다름.

| run | 바꾼 것 |
|---|---|
| v3_yolo26s_e20_s1 | seed 1 |
| v3_yolo26s_e20_s2 | seed 2 |
| v3_yolo26s_e20_s0_nomosaic | close_mosaic 20 (모자이크 계속 끔) |

In [ ]:
# 2차 v3 e20 설정에서 하나씩만 바꿔서 학습, 채점
EXTRA = {
    'v3_yolo26s_e20_s1': dict(seed=1),
    'v3_yolo26s_e20_s2': dict(seed=2),
    'v3_yolo26s_e20_s0_nomosaic': dict(close_mosaic=20),
}

for name, extra in EXTRA.items():
    run_dir = SAVE_DIR / name
    if (run_dir / 'scores.json').exists():
        print(f'{name}: 채점까지 끝나서 건너뜀')
        continue
    results_csv = run_dir / 'results.csv'
    if not (results_csv.exists() and len(results_csv.read_text().splitlines()) - 1 >= BASE_EPOCHS):
        last = run_dir / 'weights' / 'last.pt'   # 있으면 끊긴 run
        model = YOLO(last if last.exists() else f'{MODEL}.pt')
        tags = ['exp3', VARIANT, 'extra']
        model.add_callback('on_train_start', lambda trainer: tag_wandb(tags))
        if last.exists():
            model.train(resume=True)
        else:
            model.train(data=f'{DS}/data.yaml', epochs=BASE_EPOCHS, project='pill-object-detection',
                        save_dir=str(run_dir), name=name, **{**TRAIN_ARGS, **extra})
        a = model.trainer.args
        print(f'{name}: seed {a.seed}, close_mosaic {a.close_mosaic}, epochs {a.epochs}, lr0 {a.lr0}')   # 바꾼 값이 들어갔는지
    results = YOLO(run_dir / 'weights' / 'best.pt').predict(f'{DS}/images/val', conf=0.001, stream=True, verbose=False)
    preds = predictions_from_ultralytics(results)
    save_scores(run_dir, evaluate_val(preds, f'{DS}/eval/val_gt.json'), preds,
                {'condition': name, 'epochs': BASE_EPOCHS, **extra})

#### 결과: seed 반복, 모자이크 분리 (val)

| run | val | kaggle 47장 | test_only | rare_ap | frequent_ap | gap |
|---|---|---|---|---|---|---|
| 2차 기준 v3_yolo26s_e20_s0 | 0.9918 | 0.9905 | 0.9909 | 0.9915 | 0.9920 | +0.0005 |
| v3_yolo26s_e20_s1 | 0.9920 | 0.9950 | 0.9914 | 0.9920 | 0.9920 | 0.0000 |
| v3_yolo26s_e20_s2 | 0.9922 | 0.9949 | 0.9926 | 0.9913 | 0.9931 | +0.0017 |
| v3_yolo26s_e20_s0_nomosaic | 0.9944 | 0.9888 | 0.9945 | 0.9946 | 0.9942 | -0.0004 |

- 세 run 다 설정대로 들어감 (seed, close_mosaic, 20에폭, lr0 확인)
- val은 0.992~0.994로 2차 기준이랑 0.003 안. 이번에도 val로는 못 가름
- 판단은 Kaggle로

In [ ]:
# 추가 실험 3개 제출 파일 만들고 제출
EXTRA_RUNS = list(EXTRA)
for name in EXTRA_RUNS:
    out = SUB_DIR / f'exp3_{name}.csv'
    assert not out.exists(), f'{out.name} 이미 있음. 덮어쓰지 않고 멈춤'
    results = YOLO(SAVE_DIR / name / 'weights' / 'best.pt').predict(TEST_DIR, conf=0.01, device=0, stream=True, verbose=False)
    n = write_submission(predictions_from_ultralytics(results), out, COMPETITION)
    print(f'{name}: {n}줄 → {out.name}')
    run_kaggle(['submit', '-c', COMP, '-f', str(out), '-m', f'exp3 {name}'])

#### 결과: seed 반복, 모자이크 분리 (Kaggle)

| run | Kaggle public |
|---|---|
| v3 e20 seed 0 (2차) | 0.6339 |
| v3 e20 seed 1 | 0.6145 |
| v3 e20 seed 2 | 0.6243 |
| v3 e20 seed 0, 모자이크 끔 | 0.6159 |

- v3 e20 seed 3개: 평균 0.6242, 범위 0.019. 1차에서 잰 흔들림 0.02랑 같은 수준
- 2차 0.6339는 seed 범위 위쪽 끝. v3 e20 실제 수준은 0.62 안팎
- 모자이크 끈 run도 seed 범위 안이라 모자이크 효과는 못 가림
- 3차 다시 보면: both 0.6198, weight 0.6126은 seed 범위 안이라 차이 없음. sampling 0.5985만 범위보다 낮아서 나빠졌다고 볼 만함
- 섹션 4 결과의 "세 조건 다 낮음"은 이 기준으로 고쳐 읽기

## 6. TTA (v3 e20, 학습 없이 예측만)
YOLO26은 end2end 구조라 ultralytics의 `augment=True`가 안 먹음(경고만 띄우고 원본 한 번으로 예측). 그래서 직접 함.
원본이랑 좌우 뒤집은 사진을 각각 예측하고, 뒤집은 쪽 박스를 원래 위치로 되돌린 뒤 WBF로 합침.
val에서 IoU 0.95 AP가 오르면 그때만 Kaggle에 제출.

In [ ]:
!pip install -q ensemble-boxes==1.0.9
import cv2, contextlib, io
import numpy as np
from ensemble_boxes import weighted_boxes_fusion
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pill_detection.evaluate import load_gt, drop_ignored, IOU_THRESHOLDS

def make_flipped(src_dir, dst_dir):
    # 좌우로 뒤집은 사진을 같은 파일 이름으로 저장. 원래 크기(W, H)도 같이 돌려줌
    dst_dir.mkdir(parents=True, exist_ok=True)
    sizes = {}
    for p in sorted(src_dir.iterdir()):
        img = cv2.imread(str(p))
        sizes[p.name] = (img.shape[1], img.shape[0])
        cv2.imwrite(str(dst_dir / p.name), cv2.flip(img, 1))
    return sizes

def unflip(preds, sizes):
    # 뒤집은 사진 기준 박스를 원래 사진 기준으로 (x만 반대로)
    out = []
    for p in preds:
        x, y, w, h = p['bbox']
        out.append({**p, 'bbox': [sizes[p['file_name']][0] - x - w, y, w, h]})
    return out

def merge_wbf(pred_lists, sizes, iou_thr=0.55, skip_box_thr=0.001):
    # 사진마다 여러 예측 결과를 WBF로 합침. 한쪽에서만 나온 박스는 점수가 반으로 줄어듦
    by_file = {}
    for k, preds in enumerate(pred_lists):
        for p in preds:
            by_file.setdefault(p['file_name'], [[] for _ in pred_lists])[k].append(p)
    merged = []
    for name, groups in by_file.items():
        W, H = sizes[name]
        boxes = [[[max(0, x / W), max(0, y / H), min(1, (x + w) / W), min(1, (y + h) / H)]
                  for x, y, w, h in (p['bbox'] for p in g)] for g in groups]
        scores = [[p['score'] for p in g] for g in groups]
        labels = [[p['category_id'] for p in g] for g in groups]
        b, s, l = weighted_boxes_fusion(boxes, scores, labels, iou_thr=iou_thr, skip_box_thr=skip_box_thr, conf_type='avg')
        for (x1, y1, x2, y2), sc, lb in zip(b, s, l):
            merged.append({'file_name': name, 'category_id': int(round(lb)),
                           'bbox': [float(x1 * W), float(y1 * H), float((x2 - x1) * W), float((y2 - y1) * H)],
                           'score': float(sc)})
    return merged

GT = load_gt(f'{DS}/eval/val_gt.json')
COMP_IDS = {c['id'] for c in GT['categories'] if c['competition']}

def ap_by_iou(preds):
    # IoU 기준별 평균 AP (2차 14-3이랑 같은 계산)
    preds = drop_ignored(preds, GT)
    image_of = {im['file_name']: im['id'] for im in GT['images']}
    results = [{'image_id': image_of[p['file_name']], 'category_id': p['category_id'],
                'bbox': list(p['bbox']), 'score': p['score']} for p in preds]
    extra = sorted({r['category_id'] for r in results} - {c['id'] for c in GT['categories']})
    g = {**GT, 'categories': [{'id': c['id']} for c in GT['categories']] + [{'id': c} for c in extra]}
    with contextlib.redirect_stdout(io.StringIO()):
        cg = COCO(); cg.dataset = g; cg.createIndex()
        E = COCOeval(cg, cg.loadRes(results), 'bbox')
        E.params.iouThrs = IOU_THRESHOLDS
        E.params.catIds = sorted({a['category_id'] for a in GT['annotations']} & COMP_IDS)
        E.evaluate(); E.accumulate()
    prec = E.eval['precision'][:, :, :, 0, -1]   # [IoU 기준, recall, 클래스]
    per_iou = [np.mean([v[v > -1].mean() if (v > -1).any() else 0.0 for v in prec[t].T]) for t in range(len(IOU_THRESHOLDS))]
    return {f'AP{round(t * 100)}': float(a) for t, a in zip(IOU_THRESHOLDS, per_iou)}

In [ ]:
BASE_RUN = 'v3_yolo26s_e20_s0'
tta_model = YOLO(SAVE_DIR / BASE_RUN / 'weights' / 'best.pt')
val_dir = Path(DS) / 'images' / 'val'
flip_dir = Path(DS) / 'images_val_flip'

sizes = make_flipped(val_dir, flip_dir)
p_orig = predictions_from_ultralytics(tta_model.predict(str(val_dir), conf=0.001, stream=True, verbose=False))
p_flip = unflip(predictions_from_ultralytics(tta_model.predict(str(flip_dir), conf=0.001, stream=True, verbose=False)), sizes)
p_tta = merge_wbf([p_orig, p_flip], sizes)

rows = {}
for label, preds in {'원본': p_orig, '뒤집기만': p_flip, 'TTA (합침)': p_tta}.items():
    s = evaluate_val(preds, f'{DS}/eval/val_gt.json')
    rows[label] = {'val': s['val'], 'kaggle 47장': s['kaggle'], **ap_by_iou(preds)}
display(pd.DataFrame(rows).T.round(4))

#### 결과: TTA (val, v3 e20 seed 0)

| 예측 | val | kaggle 47장 | AP75 | AP80 | AP85 | AP90 | AP95 |
|---|---|---|---|---|---|---|---|
| 원본 | 0.9918 | 0.9905 | 1.0 | 1.0 | 0.9993 | 0.9979 | 0.9617 |
| 뒤집기만 | 0.9922 | 0.9938 | 1.0 | 1.0 | 0.9992 | 0.9981 | 0.9637 |
| TTA (합침) | 0.9937 | 0.9976 | 1.0 | 1.0 | 0.9992 | 0.9982 | 0.9709 |

- 원본은 2차 v3 e20 점수랑 같음. 계산 맞음
- 뒤집기만도 원본이랑 비슷해서 좌표 되돌리기 맞음
- TTA로 AP95가 0.9617 → 0.9709 (+0.009). 박스 정밀도 쪽에서 오름. 0.90 이하는 원래 거의 만점이라 그대로
- val에서 올랐으니 Kaggle 제출

In [ ]:
# test TTA 제출 파일 만들기 (원본 + 좌우 뒤집기 → WBF). 이미 있으면 건너뜀
out = SUB_DIR / f'exp3_{BASE_RUN}_tta.csv'
if out.exists():
    print(f'{out.name} 이미 있음, 건너뜀')
else:
    test_flip_dir = Path('/content/test_images_flip')
    test_sizes = make_flipped(Path(TEST_DIR), test_flip_dir)
    t_orig = predictions_from_ultralytics(tta_model.predict(TEST_DIR, conf=0.01, device=0, stream=True, verbose=False))
    t_flip = unflip(predictions_from_ultralytics(tta_model.predict(str(test_flip_dir), conf=0.01, device=0, stream=True, verbose=False)), test_sizes)
    n = write_submission(merge_wbf([t_orig, t_flip], test_sizes), out, COMPETITION)
    print(f'test 사진 {len(test_sizes)}장 | {n}줄 → {out.name}')

In [ ]:
COMP = 'ai15-level1-project'   # 덮어써진 대회 이름 되돌리기

r = subprocess.run(['kaggle', 'competitions', 'submissions', '-c', COMP, '--csv'], capture_output=True, text=True)
subs = pd.read_csv(io.StringIO(r.stdout[r.stdout.index('fileName'):]))
done = subs['fileName'].str.contains('tta').any()
print('TTA 제출 이미 있음' if done else 'TTA 제출 없음 → 지금 제출')

if not done:
    out = SUB_DIR / f'exp3_{BASE_RUN}_tta.csv'   # 06:57에 만든 파일 그대로 씀
    run_kaggle(['submit', '-c', COMP, '-f', str(out), '-m', f'exp3 {BASE_RUN} TTA (원본+좌우뒤집기, WBF)'])

#### 결과: TTA (Kaggle)

| 예측 (v3 e20 seed 0) | val | val AP95 | Kaggle public |
|---|---|---|---|
| 원본 | 0.9918 | 0.9617 | 0.6339 |
| TTA (원본+좌우뒤집기, WBF) | 0.9937 | 0.9709 | 0.6158 |

- 같은 모델인데 Kaggle -0.018. val AP95는 올랐는데 test에선 반대
- val에서 좋아진 게 test로 안 이어지는 건 2차랑 같은 패턴. 원인은 여기서도 모름
- TTA는 안 씀
- 처음 실행 때 셀 1에서 대회 이름 변수(COMP)를 클래스 목록으로 덮어써서 제출이 안 나갔음. COMP_IDS로 이름 바꾸고 다시 제출

## 7. 정리

| 실험 | run | Kaggle public |
|---|---|---|
| 2차 기준 v3 e20 | seed 0 / 1 / 2 | 0.6339 / 0.6145 / 0.6243 (평균 0.6242) |
| 3차 sampling (RFS) | v3_yolo26s_rfs | 0.5985 |
| 3차 weight (cls_pw 0.5) | v3_yolo26s_cls0.5 | 0.6126 |
| 3차 both | v3_yolo26s_rfs_cls0.5 | 0.6198 |
| 모자이크 끔 | v3_yolo26s_e20_s0_nomosaic | 0.6159 |
| TTA (seed 0) | v3_yolo26s_e20_s0 + TTA | 0.6158 |

- v3 e20은 seed만 바꿔도 0.019 흔들림. 2차 0.6339는 위쪽 끝이고 실제 수준은 0.62 안팎
- 불균형 대응: weight, both는 seed 범위 안이라 차이 없음. sampling만 범위보다 낮음. val에서 처음부터 희소 클래스가 약하지 않았음 (2차 기준 gap +0.0005)
- 모자이크: seed 범위 안이라 효과 못 가림
- TTA: val AP95는 올랐는데(0.9617 → 0.9709) Kaggle은 같은 모델 원본보다 -0.018
- val은 모든 run이 0.992~0.994로 포화. 순서도 Kaggle이랑 안 맞음
- YOLO26s 최종 후보는 v3 e20 seed 0 원본(0.6339). public 점수로 고른 거라 private에선 다른 seed랑 비슷할 수 있음